In [0]:
# Import standard libraries
import math
from collections import Counter

# Import third-party libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


seed = 24601

google_analytics = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_google_analytics_abandonment.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

sales = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_sales.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

materials = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_material.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

customer = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_customer.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

cutoff_times = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_cutoff_times.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

operating_hours = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_operating_hours.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

orders = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_orders.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

visit_plan = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_visit_plan.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

# Standardize column names to lowercase across all dataframes
for df in [
    customer,
    cutoff_times,
    google_analytics,
    materials,
    operating_hours,
    orders,
    sales,
    visit_plan,
]:
    df.columns = df.columns.str.lower()

# Helper Functions

In [0]:
def analyze_avg_time_between(purchvar, category, event_name):
    """Analyze average time between precursor event and purchase."""
    # Clean datatypes
    purch = purchvar[["event_ts_utc", "purchase_segment"]].copy()
    upd = category[["event_ts_utc", "purchase_segment"]].copy()

    purch["event_ts_utc"] = pd.to_datetime(purch["event_ts_utc"], utc=True)
    upd["event_ts_utc"] = pd.to_datetime(upd["event_ts_utc"], utc=True)
    purch["purchase_segment"] = purch["purchase_segment"].astype(str)
    upd["purchase_segment"] = upd["purchase_segment"].astype(str)

    # Merge purchases and precursors
    merged = purch.merge(
        upd,
        on="purchase_segment",
        suffixes=("_purch", f"_{event_name}"),
    )

    # Keep only valid pairs where precursor came before purchase
    merged = merged.loc[
        merged["event_ts_utc_purch"] > merged[f"event_ts_utc_{event_name}"]
    ].copy()

    # Enforce same-year rule
    same_year = (
        merged["event_ts_utc_purch"].dt.year
        == merged[f"event_ts_utc_{event_name}"].dt.year
    )
    merged = merged.loc[same_year]

    # Compute time differences in minutes
    merged["mins_between"] = (
        (merged["event_ts_utc_purch"] - merged[f"event_ts_utc_{event_name}"])
        .dt.total_seconds()
        / 60
    )

    # Compute average time gap per segment
    out = (
        merged.groupby("purchase_segment", as_index=False)
        .agg(
            avg_mins_between=("mins_between", "mean"),
            num_pairs=("mins_between", "size"),
        )
        .sort_values("avg_mins_between", ascending=False)
    )

    # Print summary
    mean_gap = out["avg_mins_between"].mean()
    print(f"{event_name}: {len(out)} segments analyzed, mean gap = {mean_gap:.2f} min")

    return out


def top_counts(series, top_n=15):
    """Get top value counts with percentages."""
    s = series.value_counts(dropna=False)
    total = s.sum()
    s = s.head(top_n)
    return pd.DataFrame(
        {
            "pattern": s.index.to_list(),
            "count": s.values,
            "percent": (s.values / total * 100) if total else [0.0] * len(s),
        }
    )


def stream_ngrams_count(seqs, n):
    """Count n-grams efficiently in sequences."""
    c = Counter()
    for seq in seqs:
        if isinstance(seq, list) and len(seq) >= n:
            # Update directly from generator
            c.update(tuple(seq[i:i + n]) for i in range(len(seq) - n + 1))
    return c


def top_ngrams_df(seqs, n=2, top_n=15):
    """Return DataFrame of top n-grams."""
    c = stream_ngrams_count(seqs, n)
    total = sum(c.values())
    rows = [
        (k, v, (v / total * 100 if total else 0.0))
        for k, v in c.most_common(top_n)
    ]
    return pd.DataFrame(rows, columns=["pattern", "count", "percent"])


def compare_tables(df_a, df_b, label_a="abandoned", label_b="completed"):
    """Compare two summary tables."""
    a = df_a.rename(
        columns={"count": f"count_{label_a}", "percent": f"percent_{label_a}"}
    )
    b = df_b.rename(
        columns={"count": f"count_{label_b}", "percent": f"percent_{label_b}"}
    )

    # Merge small top-N tables
    return a.merge(b, on="pattern", how="outer")


def describe_events(out, summary_table, event_name):
    """Describe timing distribution for an event."""
    print(out["avg_mins_between"].describe())

    perc_75 = out["avg_mins_between"].describe()["75%"]
    print(f"75% of purchases occurred in < {math.ceil(perc_75)} minute(s).")

    perc_85 = out["avg_mins_between"].quantile(0.85)
    print(f"85% of purchases occurred in < {math.ceil(perc_85)} minute(s).")

    perc_90 = out["avg_mins_between"].quantile(0.9)
    print(f"90% of purchases occurred in < {math.ceil(perc_90)} minute(s).")

    summary_table.loc[summary_table["pattern"] == event_name, "75%"] = perc_75
    summary_table.loc[summary_table["pattern"] == event_name, "85%"] = perc_85


def seg_status(s: pd.Series) -> str:
    """Determine segment-level status."""
    sl = s.dropna().astype(str).str.lower()
    if (sl == "recovered").any():
        return "recovered"
    if (sl == "intermediate").any():
        return "intermediate"
    return "not applicable"


def assign_groups(g: pd.DataFrame) -> pd.DataFrame:
    """Assign group IDs for consecutive segments."""
    rec_flag = (g["status"] == "recovered").astype(int)
    grp = rec_flag.iloc[::-1].cumsum().iloc[::-1]
    g = g.copy()
    g["group_id"] = grp
    return g


def collapse_runs(events):
    """Collapse consecutive duplicate events."""
    out, prev = [], object()
    for e in events:
        if e != prev:
            out.append(e)
        prev = e
    return out


def analyze_device(events, columnName, device_label):
    """Analyze recovery timing by device."""
    df = events[events[columnName] == device_label]

    # Group and summarize
    seg = (
        df.groupby(["customer_id", "purchase_segment"], as_index=False)
        .agg(
            start_ts=("event_ts_utc", "min"),
            end_ts=("event_ts_utc", "max"),
            status=("recovered", seg_status),
        )
    )
    seg = seg.sort_values(["customer_id", "start_ts"])
    seg = seg.groupby("customer_id", group_keys=False).apply(assign_groups)
    seg_kept = seg[seg["group_id"] > 0].copy()

    # Merge labels back
    df_labeled = df.merge(
        seg_kept[["customer_id", "purchase_segment", "group_id", "status"]],
        on=["customer_id", "purchase_segment"],
        how="inner",
    )

    # Build event sequences
    seq_df = (
        df_labeled.sort_values(["customer_id", "group_id", "event_ts_utc"])
        .groupby(["customer_id", "group_id"], as_index=False)
        .agg(
            segments=("purchase_segment", lambda s: list(pd.unique(s))),
            group_status=("status", "last"),
            start_ts=("event_ts_utc", "min"),
            end_ts=("event_ts_utc", "max"),
            num_events=("event_name", "size"),
            sequence=("event_name", list),
        )
    )
    seq_df["sequence_collapsed"] = seq_df["sequence"].apply(collapse_runs)

    # Compute durations
    seq_df["start_ts"] = pd.to_datetime(seq_df["start_ts"], utc=True, errors="coerce")
    seq_df["end_ts"] = pd.to_datetime(seq_df["end_ts"], utc=True, errors="coerce")
    seq_df["duration"] = seq_df["end_ts"] - seq_df["start_ts"]
    seq_df["duration_hours"] = seq_df["duration"].dt.total_seconds() / 3600
    dur = seq_df["duration_hours"].dropna().to_numpy()
    dur = dur[dur >= 0]

    # Print duration summary
    print(f"\nDuration summary (hours) for {device_label}")
    print(seq_df["duration_hours"].describe())

    # Plot ECDF
    p99 = np.percentile(dur, 99) if dur.size else 0.0
    xmax = p99 if np.isfinite(p99) and p99 > 0 else (dur.max() if dur.size else 1.0)
    x = np.sort(dur)
    y = np.arange(1, len(x) + 1) / len(x)
    plt.figure()
    plt.step(x, y, where="post")
    plt.xlim(0, 7000)
    plt.xlabel("Time to recovery (hours)")
    plt.ylabel("P(recovered by time ≤ t)")
    plt.title(f"Empirical chance of recovery over time ({device_label})")
    plt.grid(True)
    plt.show()

    # Compute landmark probabilities
    landmarks = [1, 6, 12, 24, 48, 72]
    landmark_probs = {f"{h}h": float((dur <= h).mean()) for h in landmarks}
    out = pd.Series(landmark_probs, name=device_label).to_frame().T
    return out


def plot_recovery_summary(recovery_summary, title_suffix):
    """Plot multi-line and grouped-bar recovery summaries."""
    cols = ["1h", "6h", "12h", "24h", "48h", "72h"]
    recovery_summary = recovery_summary[cols].astype(float)

    # Plot line chart
    x_hours = [1, 6, 12, 24, 48, 72]
    fig, ax = plt.subplots(figsize=(8, 5))
    for platform, row in recovery_summary.iterrows():
        ax.plot(x_hours, row.values, marker="o", linewidth=2, label=platform)
    ax.set_xticks(x_hours, cols)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Time horizon")
    ax.set_ylabel("P(recovered by ≤ t)")
    ax.set_title(f"Recovery Probability {title_suffix}")
    ax.grid(True, alpha=0.3)
    ax.legend(title="Platform", ncol=2, fontsize=9)
    plt.tight_layout()
    plt.show()

    # Plot grouped bar chart
    fig, ax = plt.subplots(figsize=(9, 5))
    bar_df = recovery_summary.T
    bar_df.plot(kind="bar", ax=ax)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Time horizon")
    ax.set_ylabel("P(recovered by ≤ t)")
    ax.set_title(f"Recovery Probability {title_suffix} (Grouped Bars)")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


def collapse_runs(events):
    """Collapse consecutive duplicate events."""
    collapsed = []
    previous = object()
    for event in events:
        if event != previous:
            collapsed.append(event)
        previous = event
    return collapsed


# Preparation

In [0]:
# Change the datatypes
google_analytics['event_date'] = pd.to_datetime(google_analytics['event_date'])
google_analytics['event_ts_utc'] = pd.to_datetime(google_analytics['event_ts_utc'], utc=True)

# drop NAs in the dataset
google_analytics = google_analytics.dropna(subset=['abandoned'])

# Filter out inactive Google Analytics events
events_simplified = google_analytics[
    ~(
        (google_analytics['abandoned'] == False)
        & (google_analytics['false_by_purchase'] == 'no purchase')
    )
]

# Combine event name and page name for specific event types
mask = events_simplified['event_name'].isin(['page_view', 'button_click'])
events_simplified.loc[mask, 'event_name'] = (
    events_simplified.loc[mask, 'event_name'].fillna('')
    + ' - '
    + events_simplified.loc[mask, 'event_page_name'].fillna('')
)

# Sort events by purchase segment and timestamp for sequence mining
events_sequence_mining = events_simplified.sort_values(
    ['purchase_segment', 'event_ts_utc']
)

# Build event sequences for each purchase segment
seq_df_p1 = (
    events_sequence_mining.groupby('purchase_segment')
    .agg(
        customer_id=('customer_id', 'first'),
        abandoned=('abandoned', 'first'),
        sequence=('event_name', list),
        start_ts=('event_ts_utc', 'min'),
        end_ts=('event_ts_utc', 'max'),
        num_events=('event_name', 'size')
    )
    .reset_index()
)

# Apply run-collapsing transformation to sequences
seq_df_p1['sequence_collapsed'] = seq_df_p1['sequence'].apply(collapse_runs)

In [0]:
# Compute recovery percentage among abandoned carts
events_flat = google_analytics.groupby(
    'purchase_segment', as_index=False
).last()

num_abandoned = events_flat['abandoned'].sum()
num_recovered = (events_flat['recovered'] == 'recovered').sum()

if num_abandoned > 0:
    pct = (num_recovered / num_abandoned) * 100
    print(f'{pct:.2f}% (recovered of abandoned carts)')
else:
    print('0.00% (recovered of abandoned carts; no abandoned carts found)')


# Select recovered or intermediate sessions and normalize event labels
events_recovered = google_analytics[
    google_analytics['recovered'].isin(['recovered', 'intermediate'])
].copy()

mask = events_recovered['event_name'].isin(['page_view', 'button_click'])
events_recovered.loc[mask, 'event_name'] = (
    events_recovered.loc[mask, 'event_name'].fillna('') + ' - ' +
    events_recovered.loc[mask, 'event_page_name'].fillna('')
)


# Aggregate segment-level timelines and statuses
ga = events_recovered.copy()

seg = (
    ga.groupby(['customer_id', 'purchase_segment'], as_index=False)
      .agg(
          start_ts=('event_ts_utc', 'min'),
          end_ts=('event_ts_utc', 'max'),
          status=('recovered', seg_status)
      )
)


# Order segments chronologically within each customer
seg = seg.sort_values(['customer_id', 'start_ts'])
seg = seg.groupby('customer_id', group_keys=False).apply(assign_groups)


# Keep only groups anchored by recovery (group_id > 0)
seg_kept = seg[seg['group_id'] > 0].copy()


# Attach group labels and statuses back to event-level data
ga_labeled = ga.merge(
    seg_kept[['customer_id', 'purchase_segment', 'group_id', 'status']],
    on=['customer_id', 'purchase_segment'],
    how='inner'
)


# Build per-group sequences and summary statistics
seq_df_p2 = (
    ga_labeled
      .sort_values(['customer_id', 'group_id', 'event_ts_utc'])
      .groupby(['customer_id', 'group_id'], as_index=False)
      .agg(
          segments=('purchase_segment', lambda s: list(pd.unique(s))),
          group_status=('status', 'last'),
          start_ts=('event_ts_utc', 'min'),
          end_ts=('event_ts_utc', 'max'),
          num_events=('event_name', 'size'),
          sequence=('event_name', list),
      )
)


# Collapse consecutive duplicate events in the sequence
seq_df_p2['sequence_collapsed'] = seq_df_p2['sequence'].apply(collapse_runs)

# Modeling

In [0]:
# Split the dataframe into abandoned and completed sessions
mask_abd = seq_df_p1['abandoned'].astype(bool)
seqs_abd = seq_df_p1.loc[mask_abd, 'sequence']
seqs_cmp = seq_df_p1.loc[~mask_abd, 'sequence']

# Compute first and last events for each session type
first_abd = top_counts(seqs_abd.str[0], top_n=10)
first_cmp = top_counts(seqs_cmp.str[0], top_n=10)
last_abd = top_counts(seqs_abd.str[-1], top_n=10)
last_cmp = top_counts(seqs_cmp.str[-1], top_n=10)

first_events = compare_tables(first_abd, first_cmp)
last_events = compare_tables(last_abd, last_cmp)

# Compute top full sequences (lists converted to tuples)
full_abd = top_counts(
    seqs_abd.map(lambda s: tuple(s) if isinstance(s, list) else ()),
    top_n=10
)
full_cmp = top_counts(
    seqs_cmp.map(lambda s: tuple(s) if isinstance(s, list) else ()),
    top_n=10
)
full_sequences = compare_tables(full_abd, full_cmp)

# Compute bigrams and trigrams across sessions
bigrams_abd = top_ngrams_df(seqs_abd, n=2, top_n=10)
bigrams_cmp = top_ngrams_df(seqs_cmp, n=2, top_n=10)
trigrams_abd = top_ngrams_df(seqs_abd, n=3, top_n=10)
trigrams_cmp = top_ngrams_df(seqs_cmp, n=3, top_n=10)

bigrams = compare_tables(bigrams_abd, bigrams_cmp)
trigrams = compare_tables(trigrams_abd, trigrams_cmp)

# Display summaries for inspection
print('\nTop FIRST events (abandoned vs completed):')
display(first_events.head(15))

print('\nTop LAST events (abandoned vs completed):')
display(last_events.head(15))

print('\nTop FULL sequences (abandoned vs completed):')
display(full_sequences.head(10))

print('\nTop BIGRAMS (abandoned vs completed):')
display(bigrams.head(15))

print('\nTop TRIGRAMS (abandoned vs completed):')
display(trigrams.head(15))

In [0]:
# Create a summary table from the last events containing abandonment percentages
summary_table = last_events.loc[
    last_events['percent_abandoned'].notna(),
    ['pattern', 'percent_abandoned']
].copy()

# Add empty columns for the 75% and 85% thresholds
summary_table['75%'] = ''
summary_table['85%'] = ''

# Display the first 10 rows of the summary table
summary_table.head(n=10)

In [0]:
# Create the purchases table
purchases = events_simplified.loc[
    (events_simplified['event_name'] == 'purchase')
    & (events_simplified['abandoned'] != True)
    & (events_simplified['recovered'] != 'recovered'),
    ['event_name', 'event_ts_utc', 'purchase_segment']
]

# Analyze update_cart events
update_carts = events_simplified.loc[
    events_simplified['event_name'] == 'update_cart',
    ['event_name', 'event_ts_utc', 'purchase_segment']
]
out = analyze_avg_time_between(purchases, update_carts, 'update_cart')
describe_events(out, summary_table, 'update_cart')

# Analyze remove_from_cart events
remove_from_cart = events_simplified.loc[
    events_simplified['event_name'] == 'remove_from_cart',
    ['event_name', 'event_ts_utc', 'purchase_segment']
]
out = analyze_avg_time_between(purchases, remove_from_cart, 'remove_from_cart')
describe_events(out, summary_table, 'remove_from_cart')

# Analyze view_item_list events
view_item_list = events_simplified.loc[
    events_simplified['event_name'] == 'view_item_list',
    ['event_name', 'event_ts_utc', 'purchase_segment']
]
out = analyze_avg_time_between(purchases, view_item_list, 'view_item_list')
describe_events(out, summary_table, 'view_item_list')

# Analyze page_view - Unknown Page events
unknown = events_simplified.loc[
    events_simplified['event_name'] == 'page_view - Unknown Page',
    ['event_name', 'event_ts_utc', 'purchase_segment']
]
out = analyze_avg_time_between(purchases, unknown, 'page_view - Unknown Page')
describe_events(out, summary_table, 'page_view - Unknown Page')

# Analyze page_view - Mycoke Dashboard events
mycoke_dashboard = events_simplified.loc[
    events_simplified['event_name'] == 'page_view - Mycoke Dashboard',
    ['event_name', 'event_ts_utc', 'purchase_segment']
]
out = analyze_avg_time_between(purchases, mycoke_dashboard,
                               'page_view - Mycoke Dashboard')
describe_events(out, summary_table, 'page_view - Mycoke Dashboard')

# Analyze user_engagement events
user_engagement = events_simplified.loc[
    events_simplified['event_name'] == 'user_engagement',
    ['event_name', 'event_ts_utc', 'purchase_segment']
]
out = analyze_avg_time_between(purchases, user_engagement, 'user_engagement')
describe_events(out, summary_table, 'user_engagement')

# Analyze button_click - Mycoke Orders - Cart events
orders_cart = events_simplified.loc[
    events_simplified['event_name'] == 'button_click - Mycoke Orders - Cart',
    ['event_name', 'event_ts_utc', 'purchase_segment']
]
out = analyze_avg_time_between(purchases, orders_cart,
                               'button_click - Mycoke Orders - Cart')
describe_events(out, summary_table, 'button_click - Mycoke Orders - Cart')

# Analyze page_view - Mycoke Orders events
orders = events_simplified.loc[
    events_simplified['event_name'] == 'page_view - Mycoke Orders',
    ['event_name', 'event_ts_utc', 'purchase_segment']
]
out = analyze_avg_time_between(purchases, orders, 'page_view - Mycoke Orders')
describe_events(out, summary_table, 'page_view - Mycoke Orders')

# Analyze button_click - Mycoke Orders - Checkout: Review Order events
review_order = events_simplified.loc[
    events_simplified['event_name']
    == 'button_click - Mycoke Orders - Checkout: Review Order',
    ['event_name', 'event_ts_utc', 'purchase_segment']
]
out = analyze_avg_time_between(
    purchases, review_order, 'button_click - Mycoke Orders - Checkout: Review Order'
)
describe_events(out, summary_table,
                'button_click - Mycoke Orders - Checkout: Review Order')

# Analyze proceed_to_checkout events
proceed_to_checkout = events_simplified.loc[
    events_simplified['event_name'] == 'proceed_to_checkout',
    ['event_name', 'event_ts_utc', 'purchase_segment']
]
out = analyze_avg_time_between(purchases, proceed_to_checkout,
                               'proceed_to_checkout')
describe_events(out, summary_table, 'proceed_to_checkout')

# Summarize event metrics and display results
summary_table[['75%', '85%']] = (
    summary_table[['75%', '85%']].astype(float) / 60
).round(0)
display(summary_table.head(n=10))

# Calculate 75% threshold reduction potential
total_75 = summary_table.loc[
    summary_table['75%'] < 24, 'percent_abandoned'
].sum()
print(
    "Please refer to the chart above to see which event_names correlate "
    "with a 75% chance of purchase within one day of happening. "
    "We can potentially reduce cart abandonment by",
    round(total_75, 0),
    "percent if we reach out to customers who enter any of these events "
    "and don't purchase within a day.",
)

# Calculate 85% threshold reduction potential
total_85 = summary_table.loc[
    summary_table['85%'] < 24, 'percent_abandoned'
].sum()
print(
    "Please refer to the chart above to see which event_names correlate "
    "with an 85% chance of purchase within one day of happening. "
    "We can potentially reduce cart abandonment by",
    round(total_85, 0),
    "percent if we reach out to customers who enter any of these events "
    "and don't purchase within a day.",
)

In [0]:
# Select only recovered segments
if 'group_status' in seq_df_p2.columns:
    mask_rec = (
        seq_df_p2['group_status']
        .astype(str)
        .str.lower()
        .eq('recovered')
    )
elif 'recovered' in seq_df_p2.columns:
    mask_rec = (
        seq_df_p2['recovered']
        .astype(str)
        .str.lower()
        .eq('recovered')
    )
else:
    raise KeyError(
        "seq_df_p2 requires either 'group_status' or 'recovered' column "
        "to identify recovered segments."
    )

seqs_rec = seq_df_p2.loc[mask_rec, 'sequence']

# Compute top first and last events
first_rec = top_counts(seqs_rec.str[0], top_n=15)
last_rec = top_counts(seqs_rec.str[-1], top_n=15)

# Compute top full sequences
full_rec = top_counts(
    seqs_rec.map(lambda s: tuple(s) if isinstance(s, list) else ()),
    top_n=10,
)

# Compute top bigrams and trigrams
bigrams_rec = top_ngrams_df(seqs_rec, n=2, top_n=15)
trigrams_rec = top_ngrams_df(seqs_rec, n=3, top_n=15)

# Display analysis results
print("\nTop FIRST events (recovered only):")
display(first_rec.head(15))

print("\nTop LAST events (recovered only):")
display(last_rec.head(15))

print("\nTop FULL sequences (recovered only):")
display(full_rec.head(10))

print("\nTop BIGRAMS (recovered only):")
display(bigrams_rec.head(15))

print("\nTop TRIGRAMS (recovered only):")
display(trigrams_rec.head(15))

In [0]:
# Convert timestamp columns to datetime with UTC timezone
seq_df_p2['start_ts'] = pd.to_datetime(
    seq_df_p2['start_ts'],
    utc=True,
    errors='coerce'
)
seq_df_p2['end_ts'] = pd.to_datetime(
    seq_df_p2['end_ts'],
    utc=True,
    errors='coerce'
)

# Compute duration as a timedelta
seq_df_p2['duration'] = seq_df_p2['end_ts'] - seq_df_p2['start_ts']

# Derive duration in various units
seq_df_p2['duration_hours'] = seq_df_p2['duration'].dt.total_seconds() / 3600
seq_df_p2['duration_days'] = seq_df_p2['duration'].dt.total_seconds() / 86400
seq_df_p2['duration_minutes'] = seq_df_p2['duration'].dt.total_seconds() / 60

# Display descriptive statistics for duration (in hours)
print("\nDuration summary (hours)")
print(seq_df_p2['duration_hours'].describe())

# Summarize duration by group_status
summary = (
    seq_df_p2.groupby('group_status')['duration_hours']
    .describe(percentiles=[0.25, 0.5, 0.75])
    .round(2)
)
print("\nDuration by group_status")
print(summary)

# Identify the five longest recovered sequences
longest = seq_df_p2.nlargest(
    5,
    'duration_hours'
)[['customer_id', 'group_id', 'duration_hours', 'num_events']]

# Identify the five shortest recovered sequences
shortest = seq_df_p2.nsmallest(
    5,
    'duration_hours'
)[['customer_id', 'group_id', 'duration_hours', 'num_events']]

print("\nLongest recovered sequences:")
display(longest)

print("\nShortest recovered sequences:")
display(shortest)

In [0]:
# Compute recovery duration in hours
seq_df_p2['start_ts'] = pd.to_datetime(seq_df_p2['start_ts'], utc=True, errors='coerce')
seq_df_p2['end_ts'] = pd.to_datetime(seq_df_p2['end_ts'], utc=True, errors='coerce')
seq_df_p2['duration_hours'] = (
    (seq_df_p2['end_ts'] - seq_df_p2['start_ts']).dt.total_seconds() / 3600
)

# Filter valid durations
durations = seq_df_p2.loc[seq_df_p2['duration_hours'] > 0, 'duration_hours']

# Calculate the 90th percentile of duration
p90 = durations.quantile(0.90)

# Plot histogram with 90th percentile line
plt.figure(figsize=(8, 4))
sns.histplot(durations, bins=30, kde=True, color='skyblue')
plt.axvline(
    p90,
    color='red',
    linestyle='--',
    linewidth=2,
    label=f"90th percentile = {p90:.1f} hrs",
)
plt.title("Distribution of Recovery Durations (Hours)", fontsize=13, weight='bold')
plt.xlabel("Duration (hours)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

# Recompute histogram statistics
durations = seq_df_p2.loc[seq_df_p2['duration_hours'] > 0, 'duration_hours']
counts, bin_edges = np.histogram(durations, bins=30)

# Extract first bin statistics
first_bin_count = int(counts[0])
first_bin_start = bin_edges[0]
first_bin_end = bin_edges[1]
first_bin_range = (first_bin_start, first_bin_end)

# Compute percentage of observations in the first bin
total_obs = int(counts.sum())
first_bin_pct = (first_bin_count / total_obs * 100) if total_obs else 0.0

# Print summary statistics
print(f"Total observations: {total_obs:,}")
print(
    f"Number of observations in first bin: {first_bin_count:,} "
    f"({first_bin_pct:.2f}% of total)"
)
print(
    f"Duration range of first bin: "
    f"{first_bin_range[0]:.2f} to {first_bin_range[1]:.2f} hours"
)

# Visualize histogram with first bin boundary
plt.figure(figsize=(8, 4))
sns.histplot(durations, bins=30, kde=True, color='skyblue')
plt.axvline(
    first_bin_end,
    color='green',
    linestyle='--',
    linewidth=2,
    label=(
        f"End of 1st bin = {first_bin_end:.1f} hrs\n"
        f"{first_bin_pct:.1f}% of data"
    ),
)
plt.title(
    "Histogram of Recovery Durations with First Bin Boundary",
    fontsize=13,
    weight='bold',
)
plt.xlabel("Duration (hours)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

In [0]:
# Extract valid duration values
dur = seq_df_p2['duration_hours'].dropna().to_numpy()
dur = dur[dur >= 0]

# Compute percentile cap for plotting range
p99 = np.percentile(dur, 99) if dur.size else 0.0
xmax = p99 if np.isfinite(p99) and p99 > 0 else (dur.max() if dur.size else 1.0)

# Compute empirical cumulative distribution function
x = np.sort(dur)
y = np.arange(1, len(x) + 1) / len(x)

# Plot ECDF of recovery duration
plt.figure()
plt.step(x, y, where="post")
plt.xlim(0, xmax)
plt.xlabel("Time to recovery (hours)")
plt.ylabel("P(recovered by time ≤ t)")
plt.title("Empirical chance of recovery over time")
plt.grid(True)
plt.show()

# Compute quick probability landmarks
landmarks = [1, 6, 12, 24, 48, 72]
landmark_probs = {f"{h}h": float((dur <= h).mean()) for h in landmarks}
pd.Series(landmark_probs, name="P(recovered by ≤ t)").to_frame()

In [0]:
# Analyze recovery probability by device category
desktop_tbl = analyze_device(events_recovered, 'device_category', 'Desktop')
mobile_tbl = analyze_device(events_recovered, 'device_category', 'Mobile')
tablet_tbl = analyze_device(events_recovered, 'device_category', 'Tablet')

# Combine results from all device categories into one summary table
recovery_summary = pd.concat([desktop_tbl, mobile_tbl, tablet_tbl])

# Display the combined recovery probability summary
print("\nCombined Recovery Probability Table")
display(recovery_summary)

In [0]:
# Plot the recovery summary by device category
plot_recovery_summary(recovery_summary, 'device_category')

In [0]:
# Analyze recovery probability by device brand
google = analyze_device(events_recovered, 'device_mobile_brand_name', 'Google')
apple = analyze_device(events_recovered, 'device_mobile_brand_name', 'Apple')
microsoft = analyze_device(events_recovered, 'device_mobile_brand_name', 'Microsoft')
samsung = analyze_device(events_recovered, 'device_mobile_brand_name', 'Samsung')
mozilla = analyze_device(events_recovered, 'device_mobile_brand_name', 'Mozilla')
other = analyze_device(events_recovered, 'device_mobile_brand_name', 'Other')

# Combine all device results into a single summary table
recovery_summary = pd.concat(
    [google, apple, microsoft, samsung, mozilla, other]
).reset_index(drop=True)

# Display the combined recovery probability table
print("\nCombined Recovery Probability Table")
display(recovery_summary)

In [0]:
# Plot recovery summary by mobile brand name
plot_recovery_summary(recovery_summary, 'device_mobile_brand_name')

In [0]:
# Analyze recovery probability for each device operating system
windows = analyze_device(events_recovered, 'device_operating_system', 'Windows')
ios = analyze_device(events_recovered, 'device_operating_system', 'iOS')
macintosh = analyze_device(events_recovered, 'device_operating_system', 'Macintosh')
android = analyze_device(events_recovered, 'device_operating_system', 'Android')
chrome = analyze_device(events_recovered, 'device_operating_system', 'Chrome OS')

# Combine all device-specific results into a single summary table
recovery_summary = pd.concat([windows, ios, macintosh, android, chrome])

# Display the combined recovery probability summary
print("\nCombined Recovery Probability Table")
display(recovery_summary)

In [0]:
# Plot the recovery summary grouped by device operating system
plot_recovery_summary(recovery_summary, 'device_operating_system')